# Generator reactive-power (Q) limits

DPsim's Newton-Raphson powerflow solver can enforce generator Qmin/Qmax limits with a bidirectional PV<->PQ outer loop: a PV bus whose generator reactive output exceeds its limit is pinned at the limit and the bus is converted to PQ; if the constraint later relaxes, the bus reverts to PV. Enforcement is opt-in via `set_pf_solver_enforce_q_limits(True)` on the `Simulation` (default off, so a case with no limits set behaves exactly as before).

This notebook uses a small hand-wired three-bus case (slack, a PiLine, a PV generator with a configurable Qmax, another PiLine, a PQ load) and sweeps Qmax across values that do and do not bind the generator's natural reactive output. It checks that a binding limit pins the generator exactly at Qmax, that a non-binding limit leaves the bus on voltage control at its setpoint, and that the dense and sparse Jacobian solvers agree.

In [ ]:
import math
import numpy as np
import pandas as pd
import dpsimpy

In [ ]:
BASE_V = 230e3
BASE_S = 100e6
V_SET = 1.02


def run_pf(q_max_mvar, enforce, use_sparse):
    n1 = dpsimpy.sp.SimNode("n1", dpsimpy.PhaseType.Single)
    n2 = dpsimpy.sp.SimNode("n2", dpsimpy.PhaseType.Single)
    n3 = dpsimpy.sp.SimNode("n3", dpsimpy.PhaseType.Single)

    slack = dpsimpy.sp.ph1.SynchronGenerator("slack")
    slack.set_parameters(
        rated_apparent_power=BASE_S,
        rated_voltage=BASE_V,
        set_point_active_power=0.0,
        set_point_voltage=BASE_V,
        powerflow_bus_type=dpsimpy.PowerflowBusType.VD,
    )
    slack.set_base_voltage(BASE_V)
    slack.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.VD)
    slack.connect([n1])

    gen = dpsimpy.sp.ph1.SynchronGenerator("gen")
    gen.set_parameters(
        rated_apparent_power=BASE_S,
        rated_voltage=BASE_V,
        set_point_active_power=50e6,
        set_point_voltage=V_SET * BASE_V,
        powerflow_bus_type=dpsimpy.PowerflowBusType.PV,
        q_limit_max=q_max_mvar * 1e6,
        q_limit_min=-200e6,
    )
    gen.set_base_voltage(BASE_V)
    gen.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.PV)
    gen.connect([n2])

    load = dpsimpy.sp.ph1.Load("load")
    load.set_parameters(120e6, 120e6, BASE_V)
    load.modify_power_flow_bus_type(dpsimpy.PowerflowBusType.PQ)
    load.connect([n3])

    z_base = BASE_V**2 / BASE_S
    line12 = dpsimpy.sp.ph1.PiLine("line12")
    line12.set_parameters(0.02 * z_base, 0.06 * z_base / (2 * math.pi * 50), 0.0, 0.0)
    line12.set_base_voltage(BASE_V)
    line12.connect([n1, n2])

    line23 = dpsimpy.sp.ph1.PiLine("line23")
    line23.set_parameters(0.02 * z_base, 0.06 * z_base / (2 * math.pi * 50), 0.0, 0.0)
    line23.set_base_voltage(BASE_V)
    line23.connect([n2, n3])

    system = dpsimpy.SystemTopology(
        50, [n1, n2, n3], [slack, gen, load, line12, line23]
    )

    sim = dpsimpy.Simulation("qlimit_pf", dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.set_time_step(0.1)
    sim.set_final_time(0.1)
    sim.set_domain(dpsimpy.Domain.SP)
    sim.set_solver(dpsimpy.Solver.NRP)
    sim.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Simulation)
    sim.set_pf_solver_use_sparse(use_sparse)
    sim.set_pf_solver_enforce_q_limits(enforce)
    sim.run()

    s_gen = complex(np.asarray(n2.attr("s").get()).flatten()[0])
    v_gen = abs(complex(n2.single_voltage()) / BASE_V)
    return v_gen, s_gen.imag / 1e6

In [ ]:
v_nat, q_nat = run_pf(q_max_mvar=1e9, enforce=False, use_sparse=False)

rows = []
for use_sparse in (False, True):
    for q_max in (5, 50, 150, 250):
        v, q = run_pf(q_max, enforce=True, use_sparse=use_sparse)
        rows.append(
            {
                "solver": "sparse" if use_sparse else "dense",
                "Qmax [MVAr]": q_max,
                "binding": q_max < q_nat,
                "V [pu]": v,
                "gen Q [MVAr]": q,
            }
        )
results = pd.DataFrame(rows)
results

In [ ]:
for solver in ("dense", "sparse"):
    subset = results[results["solver"] == solver]
    for _, row in subset.iterrows():
        if row["binding"]:
            assert (
                abs(row["gen Q [MVAr]"] - row["Qmax [MVAr]"]) < 0.5
            ), f"{solver}: gen not pinned at Qmax={row['Qmax [MVAr]']}"
        else:
            assert abs(row["V [pu]"] - V_SET) < 1e-3, (
                f"{solver}: PV bus not held at V_set for " f"Qmax={row['Qmax [MVAr]']}"
            )

# unlimited case (enforcement off) must be untouched by the feature
assert abs(v_nat - V_SET) < 1e-3

# dense and sparse solvers must agree
dense = results[results["solver"] == "dense"].reset_index(drop=True)
sparse = results[results["solver"] == "sparse"].reset_index(drop=True)
assert np.allclose(dense["V [pu]"], sparse["V [pu]"], atol=1e-8)
assert np.allclose(dense["gen Q [MVAr]"], sparse["gen Q [MVAr]"], atol=1e-6)

print(f"unlimited gen Q (no enforcement): {q_nat:.2f} MVAr")
print(
    "Q-limit enforcement verified: gen Q pins at the violated limit, dense == sparse."
)